# Taller 1 - EcoMarket + Olist

Chatbot de atención al cliente para EcoMarket, construido sobre el dataset
público de Olist. Este notebook cubre la **Fase 3** del taller (ingeniería
de prompts) y se ejecuta de forma local, usando un modelo de lenguaje
**open-source ejecutado localmente con Ollama** (sin costo ni claves de pago).

**Contenido**
1. Introducción
2. Fuente de datos
3. Carga de datos
4. Exploración de los datos
5. Selección de pedidos
6. Construcción de la base de pruebas
7. Prompt para consulta de pedidos
8. Prompt para devoluciones
9. Configuración del modelo de IA generativa (modelo local con Ollama)
10. Pruebas de los prompts
11. Resultados
12. Conclusiones

## 1. Introducción

EcoMarket es una empresa de e-commerce de productos sostenibles que recibe
miles de consultas diarias de soporte. El 80% son repetitivas (estado de
pedido, devoluciones) y hoy se resuelven con un tiempo de respuesta promedio
de 24 horas.

Este notebook implementa y prueba, con datos reales del dataset de Olist,
los dos prompts propuestos en la Fase 3 del taller:

- **Ejercicio 1:** consulta del estado de un pedido.
- **Ejercicio 2:** evaluación de si un producto puede devolverse.

El modelo de lenguaje utilizado es un modelo **open-source** (por ejemplo,
Llama 3.2) ejecutado localmente con **Ollama**, tal como lo permite la
rúbrica del taller para esta fase práctica.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

print("Entorno preparado correctamente")

## 2. Fuente de datos

Usamos el dataset público de [Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
como base de datos de ejemplo para simular los pedidos de EcoMarket.

El notebook se ejecuta **de forma local**: los archivos CSV deben estar
disponibles en disco (no se usa `google.colab`). Ajusta `RUTA_BASE` si
mueves la carpeta del proyecto a otra ubicación.

In [ ]:
# Rutas locales del proyecto
RUTA_BASE = Path(
    r"D:\Diego Angrino Chiran\Documentos\Maestria Ingenieria de la investigación ICESI"
    r"\Segundo semestre\IA_Generativa\Taller_1"
)
RUTA_ENTRADA = RUTA_BASE / "Entrada" / "archive"
RUTA_SALIDA = RUTA_BASE / "Salida"

RUTA_SALIDA.mkdir(parents=True, exist_ok=True)

print("Carpeta de datos de entrada:", RUTA_ENTRADA)
print("Carpeta de resultados:", RUTA_SALIDA)
print("¿Existe la carpeta de entrada?:", RUTA_ENTRADA.exists())

## 3. Carga de datos

In [ ]:
archivos = os.listdir(RUTA_ENTRADA)

print("Archivos disponibles en la carpeta de entrada:")
for archivo in archivos:
    if archivo.endswith(".csv"):
        print(archivo)

In [ ]:
archivos_csv = sorted([
    archivo for archivo in os.listdir(RUTA_ENTRADA)
    if archivo.endswith(".csv")
])

for archivo in archivos_csv:
    df = pd.read_csv(RUTA_ENTRADA / archivo)

    print("=" * 80)
    print(f"ARCHIVO: {archivo}")
    print(f"Filas: {df.shape[0]:,}")
    print(f"Columnas: {df.shape[1]}")
    print("Variables:")
    print(list(df.columns))
    print()

## 4. Exploración de los datos

In [ ]:
# Cargar la tabla principal de pedidos

pedidos = pd.read_csv(RUTA_ENTRADA / "olist_orders_dataset.csv")

print("Tabla de pedidos cargada correctamente")
print(f"Registros: {len(pedidos):,}")
print(f"Columnas: {len(pedidos.columns)}")

pedidos.head()

In [ ]:
# Estados reales de los pedidos
estados = pedidos["order_status"].value_counts()
# Traducción de los estados para visualización
traduccion_estados = {
    "delivered": "Entregado",
    "shipped": "Enviado",
    "canceled": "Cancelado",
    "unavailable": "No disponible",
    "invoiced": "Facturado",
    "processing": "En procesamiento",
    "created": "Creado",
    "approved": "Aprobado"
}
estados_espanol = estados.rename(index=traduccion_estados)
print("Estados de pedidos en español:")
print(estados_espanol)

## 5. Selección de pedidos

In [ ]:
# Seleccionar 10 pedidos reales para las pruebas

cantidades = {
    "delivered": 3,     # Entregado
    "shipped": 2,       # Enviado
    "canceled": 2,      # Cancelado
    "unavailable": 1,   # No disponible
    "invoiced": 1,      # Facturado
    "processing": 1     # En procesamiento
}

seleccionados = []

for estado, cantidad in cantidades.items():
    muestra = pedidos[pedidos["order_status"] == estado].head(cantidad)
    seleccionados.append(muestra)

pedidos_prueba = pd.concat(seleccionados, ignore_index=True)

print("Total de pedidos seleccionados:", len(pedidos_prueba))

In [ ]:
# Crear una tabla de presentación en español

pedidos_presentacion = pedidos_prueba[[
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]].copy()

# Traducir estados
pedidos_presentacion["estado_espanol"] = (
    pedidos_presentacion["order_status"]
    .map(traduccion_estados)
)

# Renombrar columnas
pedidos_presentacion = pedidos_presentacion.rename(columns={
    "order_id": "Numero_pedido",
    "order_status": "Estado_original",
    "order_purchase_timestamp": "Fecha_compra",
    "order_delivered_carrier_date": "Fecha_entrega_transportadora",
    "order_delivered_customer_date": "Fecha_entrega_cliente",
    "order_estimated_delivery_date": "Fecha_entrega_estimada",
    "estado_espanol": "Estado"
})

# Ordenar columnas
pedidos_presentacion = pedidos_presentacion[[
    "Numero_pedido",
    "Estado",
    "Estado_original",
    "Fecha_compra",
    "Fecha_entrega_transportadora",
    "Fecha_entrega_cliente",
    "Fecha_entrega_estimada"
]]

pedidos_presentacion

## 6. Construcción de la base de pruebas

In [ ]:
# Cargar información de artículos y productos

items = pd.read_csv(RUTA_ENTRADA / "olist_order_items_dataset.csv")
productos = pd.read_csv(RUTA_ENTRADA / "olist_products_dataset.csv")

print("Tabla de artículos:", items.shape)
print("Tabla de productos:", productos.shape)

In [ ]:
# Obtener los artículos correspondientes a nuestros 10 pedidos

items_prueba = items[
    items["order_id"].isin(pedidos_prueba["order_id"])
].copy()

print("Artículos encontrados para nuestros 10 pedidos:", len(items_prueba))

items_prueba

In [ ]:
# Relacionar los artículos con la información de los productos

items_productos_prueba = items_prueba.merge(
    productos,
    on="product_id",
    how="left"
)

print("Registros después de relacionar artículos y productos:",
      len(items_productos_prueba))

items_productos_prueba[[
    "order_id",
    "product_id",
    "product_category_name",
    "price",
    "freight_value"
]]

In [ ]:
# Cargar la tabla de traducción de categorías

traducciones = pd.read_csv(RUTA_ENTRADA / "product_category_name_translation.csv")

print("Registros de traducción:", len(traducciones))
print("Columnas:", list(traducciones.columns))

traducciones.head(10)

In [ ]:
# Unir las categorías en inglés de Olist

items_productos_prueba = items_productos_prueba.merge(
    traducciones,
    on="product_category_name",
    how="left"
)

print("Información de productos con traducción:")
print(
    items_productos_prueba[
        [
            "order_id",
            "product_id",
            "product_category_name",
            "product_category_name_english",
            "price",
            "freight_value"
        ]
    ].to_string(index=False)
)

In [ ]:
# Traducción de categorías al español para la presentación del taller

categorias_espanol = {
    "perfumaria": "Perfumería",
    "informatica_acessorios": "Informática y accesorios",
    "automotivo": "Automotriz",
    "moveis_decoracao": "Muebles y decoración",
    "utilidades_domesticas": "Utilidades domésticas",
    "beleza_saude": "Belleza y salud"
}

items_productos_prueba["categoria_espanol"] = (
    items_productos_prueba["product_category_name"]
    .map(categorias_espanol)
    .fillna("Sin categoría registrada")
)

items_productos_prueba[[
    "order_id",
    "product_category_name",
    "product_category_name_english",
    "categoria_espanol",
    "price",
    "freight_value"
]]

In [ ]:
# Unir los pedidos con la información de sus productos

pedidos_completos = pedidos_prueba.merge(
    items_productos_prueba,
    on="order_id",
    how="left"
)

print("Registros de la tabla final:", len(pedidos_completos))
print("Columnas disponibles:")
print(list(pedidos_completos.columns))

In [ ]:
# Crear la base de datos limpia para las pruebas del chatbot

base_chatbot = pedidos_completos[[
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
    "categoria_espanol",
    "price",
    "freight_value"
]].copy()

# Traducir el estado al español
base_chatbot["estado_espanol"] = (
    base_chatbot["order_status"]
    .map(traduccion_estados)
)

# Renombrar las columnas para facilitar su uso
base_chatbot = base_chatbot.rename(columns={
    "order_id": "numero_pedido",
    "order_status": "estado_original",
    "order_purchase_timestamp": "fecha_compra",
    "order_estimated_delivery_date": "fecha_entrega_estimada",
    "order_delivered_customer_date": "fecha_entrega_real",
    "categoria_espanol": "categoria_producto",
    "price": "precio_producto",
    "freight_value": "costo_transporte"
})

# Organizar columnas
base_chatbot = base_chatbot[[
    "numero_pedido",
    "estado_espanol",
    "estado_original",
    "fecha_compra",
    "fecha_entrega_estimada",
    "fecha_entrega_real",
    "categoria_producto",
    "precio_producto",
    "costo_transporte"
]]

base_chatbot

In [ ]:
# Verificación de los 10 registros

print("Número total de pedidos:", len(base_chatbot))

print("\nTipos de datos:")
print(base_chatbot.dtypes)

print("\nTabla completa:")
display(base_chatbot)

In [ ]:
# Guardar la base de pruebas en Excel

archivo_excel = RUTA_SALIDA / "Taller_1_EcoMarket_Datos.xlsx"

with pd.ExcelWriter(archivo_excel, engine="openpyxl") as writer:
    base_chatbot.to_excel(
        writer,
        sheet_name="Pedidos_Prueba",
        index=False
    )

print(f"Archivo creado correctamente: {archivo_excel}")

In [ ]:
# Comprobar el archivo Excel generado

archivo_excel = RUTA_SALIDA / "Taller_1_EcoMarket_Datos.xlsx"

print("¿El archivo existe?:", archivo_excel.exists())

# Leer nuevamente el Excel
verificacion = pd.read_excel(archivo_excel, sheet_name="Pedidos_Prueba")

print("\nNúmero de registros:", len(verificacion))
print("Número de columnas:", len(verificacion.columns))

print("\nContenido del Excel:")
display(verificacion)

In [ ]:
# Preparar la base de prueba para el chatbot

base_prueba = base_chatbot.copy()

# Convertir fechas a formato datetime
base_prueba["fecha_compra"] = pd.to_datetime(
    base_prueba["fecha_compra"],
    errors="coerce"
)

base_prueba["fecha_entrega_estimada"] = pd.to_datetime(
    base_prueba["fecha_entrega_estimada"],
    errors="coerce"
)

base_prueba["fecha_entrega_real"] = pd.to_datetime(
    base_prueba["fecha_entrega_real"],
    errors="coerce"
)

# Crear una respuesta esperada según el estado real
respuestas_estado = {
    "Entregado": "El pedido fue entregado.",
    "Enviado": "El pedido fue enviado y aún no registra entrega al cliente.",
    "Cancelado": "El pedido fue cancelado.",
    "No disponible": "El pedido aparece como no disponible en el registro.",
    "Facturado": "El pedido fue facturado y aún no registra entrega al cliente.",
    "En procesamiento": "El pedido se encuentra en procesamiento y aún no registra entrega al cliente."
}

base_prueba["respuesta_esperada"] = (
    base_prueba["estado_espanol"].map(respuestas_estado)
)

print("Base de prueba preparada correctamente.")
print("Registros:", len(base_prueba))
print("Columnas:", len(base_prueba.columns))

display(base_prueba)

In [ ]:
# Función básica para consultar el estado de un pedido

def consultar_pedido(numero_pedido):
    resultado = base_prueba[
        base_prueba["numero_pedido"] == numero_pedido
    ]

    if resultado.empty:
        return "No se encontró el pedido en la base de datos."

    pedido = resultado.iloc[0]

    return (
        f"Pedido: {pedido['numero_pedido']}\n"
        f"Estado: {pedido['estado_espanol']}\n"
        f"Fecha de compra: {pedido['fecha_compra']}\n"
        f"Fecha estimada de entrega: {pedido['fecha_entrega_estimada']}"
    )


# Probar con un pedido REAL de nuestra base
numero_prueba = "e481f51cbdc54678b7cc49136f2d6af7"

respuesta = consultar_pedido(numero_prueba)

print(respuesta)

## 7. Prompt para consulta de pedidos (Ejercicio 1 - Fase 3)

In [ ]:
# Prompt base para el chatbot de EcoMarket
# Fase 3 - Ejercicio 1: Consulta de estado de pedido

prompt_estado_pedido = """
Eres el asistente virtual de atención al cliente de EcoMarket.

Tu función es informar al cliente sobre el estado de su pedido
utilizando exclusivamente la información proporcionada en la
base de datos.

REGLAS:
1. No inventes información.
2. No modifiques el estado registrado del pedido.
3. Si el pedido no aparece en la base de datos, indícalo claramente.
4. Responde en español.
5. Utiliza un tono amable, claro y profesional.
6. Si existe una fecha estimada de entrega, comunícala.
7. Si no existe una fecha registrada, indica que no está disponible.
8. No proporciones información que no esté presente en los datos.

DATOS DEL PEDIDO:
{datos_pedido}

PREGUNTA DEL CLIENTE:
{pregunta_cliente}

Genera una respuesta breve y útil para el cliente.
"""

# Ejemplo utilizando nuestro pedido real
pedido = base_prueba[
    base_prueba["numero_pedido"] == "e481f51cbdc54678b7cc49136f2d6af7"
].iloc[0]

datos_pedido = pedido.to_dict()

pregunta_cliente = "¿Cuál es el estado de mi pedido?"

prompt_generado = prompt_estado_pedido.format(
    datos_pedido=datos_pedido,
    pregunta_cliente=pregunta_cliente
)

print(prompt_generado)

In [ ]:
# Respuesta esperada del asistente para el pedido real

respuesta_asistente = (
    "Hola. Tu pedido e481f51cbdc54678b7cc49136f2d6af7 "
    "se encuentra en estado Entregado. "
    "La fecha de entrega registrada fue el 10 de octubre de 2017."
)

print("Respuesta del asistente:")
print(respuesta_asistente)

In [ ]:
# Probar el chatbot con los 10 pedidos reales (respuesta esperada, sin LLM todavía)

for _, pedido in base_prueba.iterrows():

    print("=" * 80)
    print(f"PEDIDO: {pedido['numero_pedido']}")
    print(f"ESTADO REAL: {pedido['estado_espanol']}")

    if pd.notna(pedido["fecha_entrega_real"]):
        fecha_real = pedido["fecha_entrega_real"].strftime("%d/%m/%Y")
        print(f"FECHA DE ENTREGA: {fecha_real}")
    else:
        print("FECHA DE ENTREGA: No registrada")

    print(f"RESPUESTA ESPERADA: {pedido['respuesta_esperada']}")

## 8. Prompt para devoluciones (Ejercicio 2 - Fase 3)

In [ ]:
# ============================================
# EJERCICIO 2 - CASOS DE DEVOLUCIONES
# ============================================

casos_devolucion = [
    {
        "producto": "Producto de limpieza para el hogar",
        "categoria": "Utilidades domésticas",
        "motivo": "El producto llegó en buenas condiciones, pero el cliente cambió de opinión.",
        "puede_devolver": True
    },
    {
        "producto": "Perfume",
        "categoria": "Perfumería",
        "motivo": "El cliente ya abrió y utilizó el producto.",
        "puede_devolver": False
    },
    {
        "producto": "Accesorio de computadora",
        "categoria": "Informática y accesorios",
        "motivo": "El producto llegó defectuoso.",
        "puede_devolver": True
    },
    {
        "producto": "Producto de higiene personal",
        "categoria": "Belleza y salud",
        "motivo": "El producto fue abierto después de la entrega.",
        "puede_devolver": False
    },
    {
        "producto": "Mueble para el hogar",
        "categoria": "Muebles y decoración",
        "motivo": "El producto llegó con daños visibles.",
        "puede_devolver": True
    }
]

df_devoluciones = pd.DataFrame(casos_devolucion)

display(df_devoluciones)

In [ ]:
prompt_devolucion = """
Eres el asistente virtual de atención al cliente de EcoMarket.

Tu función es ayudar a los clientes a determinar si un producto
puede ser devuelto, utilizando exclusivamente las reglas de
devolución proporcionadas.

REGLAS DE DEVOLUCIÓN:

1. Los productos que llegaron defectuosos pueden ser devueltos.
2. Los productos que llegaron dañados pueden ser devueltos.
3. Los productos de higiene personal que hayan sido abiertos
   después de la entrega no pueden ser devueltos.
4. Los perfumes que hayan sido abiertos o utilizados no pueden
   ser devueltos.
5. Un producto permitido puede ser devuelto aunque el cliente
   simplemente haya cambiado de opinión.
6. Si la información proporcionada no permite determinar si el
   producto puede devolverse, debes indicarlo claramente.
7. No inventes políticas, plazos, condiciones ni excepciones
   que no estén presentes en las reglas.
8. Responde siempre en español.
9. Utiliza un tono amable, claro y empático.
10. Explica brevemente el motivo de la decisión.

INFORMACIÓN DEL PRODUCTO:
{datos_producto}

MOTIVO DE LA DEVOLUCIÓN:
{motivo_devolucion}

RESPUESTA:

Indica claramente si el producto puede devolverse o no.
Después explica brevemente el motivo y proporciona una respuesta
empática al cliente.
"""

print(prompt_devolucion)

## 9. Configuración del modelo de IA generativa (modelo local con Ollama)

Para este ejercicio se usa un modelo de lenguaje **open-source, gratuito y
100% local** mediante [Ollama](https://ollama.com/), en lugar de una API de
pago (Gemini u OpenAI). La rúbrica del taller permite explícitamente usar un
modelo open-source para esta fase práctica.

Pasos previos (una sola vez, fuera del notebook):

1. Instala Ollama desde `ollama.com/download` (Windows, Mac o Linux).
2. Abre una terminal y descarga un modelo, por ejemplo:
   `ollama pull llama3.2`
3. Ollama queda corriendo en segundo plano automáticamente tras instalarlo,
   escuchando en `http://localhost:11434`. Si esta sección falla con un
   error de conexión, abre la aplicación de Ollama o ejecuta `ollama serve`
   en una terminal.

Ollama expone una API compatible con la de OpenAI, así que seguimos usando
la librería `openai` como cliente, apuntándola a `http://localhost:11434/v1`
en lugar de a los servidores de OpenAI. No se necesita ninguna clave de pago.

In [ ]:
# Instalar la librería openai (se usa como cliente para hablar con Ollama,
# que expone una API compatible con la de OpenAI)

!pip -q install -U openai

print("Librería openai instalada correctamente.")

In [ ]:
# Ollama corre en tu propia máquina y no requiere una clave de pago real.
# La librería de OpenAI exige un valor no vacío, así que usamos un texto cualquiera.
api_key = "ollama"

print("Configuración de clave lista (no se necesita una clave de pago para usar Ollama).")

In [ ]:
import openai
from openai import OpenAI

MODELO_LOCAL = "llama3.2"  # cambia este nombre si descargaste otro modelo con 'ollama pull'

cliente = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key=api_key
)

print("Conexión con el modelo local (Ollama) configurada correctamente.")

In [ ]:
# Seleccionar un pedido real de nuestra base de datos

pedido = base_prueba[
    base_prueba["numero_pedido"] == "e481f51cbdc54678b7cc49136f2d6af7"
].iloc[0]

datos_pedido = pedido.to_dict()

pregunta_cliente = "¿Cuál es el estado de mi pedido?"

print("Pedido seleccionado:")
print(datos_pedido)

In [ ]:
# Generar una respuesta de atención al cliente con el modelo local

prompt_final = prompt_estado_pedido.format(
    datos_pedido=datos_pedido,
    pregunta_cliente=pregunta_cliente
)

try:
    respuesta = cliente.chat.completions.create(
        model=MODELO_LOCAL,
        messages=[{"role": "user", "content": prompt_final}]
    )
    print("RESPUESTA GENERADA POR EL MODELO LOCAL:")
    print(respuesta.choices[0].message.content)
except openai.APIConnectionError:
    print("No se pudo conectar con Ollama. Verifica que esté corriendo (abre la app de Ollama o ejecuta 'ollama serve').")
except openai.NotFoundError:
    print(f"El modelo '{MODELO_LOCAL}' no está descargado. Ejecuta en una terminal: ollama pull {MODELO_LOCAL}")
except openai.OpenAIError as e:
    print(f"Error al llamar al modelo local: {e}")

In [ ]:
# Probar la conexión con el modelo local

try:
    prueba = cliente.chat.completions.create(
        model=MODELO_LOCAL,
        messages=[{"role": "user", "content": "Responde únicamente: conexión exitosa"}]
    )
    print(prueba.choices[0].message.content)
except openai.APIConnectionError:
    print("No se pudo conectar con Ollama. Verifica que esté corriendo (abre la app de Ollama o ejecuta 'ollama serve').")
except openai.NotFoundError:
    print(f"El modelo '{MODELO_LOCAL}' no está descargado. Ejecuta en una terminal: ollama pull {MODELO_LOCAL}")
except openai.OpenAIError as e:
    print(f"Error al llamar al modelo local: {e}")

In [ ]:
# Generar la primera respuesta real del chatbot EcoMarket

prompt_final = prompt_estado_pedido.format(
    datos_pedido=datos_pedido,
    pregunta_cliente=pregunta_cliente
)

try:
    respuesta = cliente.chat.completions.create(
        model=MODELO_LOCAL,
        messages=[{"role": "user", "content": prompt_final}]
    )
    print("RESPUESTA GENERADA POR EL MODELO LOCAL:")
    print(respuesta.choices[0].message.content)
except openai.APIConnectionError:
    print("No se pudo conectar con Ollama. Verifica que esté corriendo (abre la app de Ollama o ejecuta 'ollama serve').")
except openai.NotFoundError:
    print(f"El modelo '{MODELO_LOCAL}' no está descargado. Ejecuta en una terminal: ollama pull {MODELO_LOCAL}")
except openai.OpenAIError as e:
    print(f"Error al llamar al modelo local: {e}")

## 10. Pruebas de los prompts - Ejercicio 1 (los 10 pedidos)

In [ ]:
resultados_chatbot = []

for _, pedido in base_prueba.iterrows():
    datos_pedido = pedido.to_dict()
    pregunta_cliente = "¿Cuál es el estado de mi pedido?"

    prompt_final = prompt_estado_pedido.format(
        datos_pedido=datos_pedido,
        pregunta_cliente=pregunta_cliente
    )

    try:
        respuesta = cliente.chat.completions.create(
            model=MODELO_LOCAL,
            messages=[{"role": "user", "content": prompt_final}]
        )
        texto_respuesta = respuesta.choices[0].message.content
    except Exception as e:
        texto_respuesta = f"ERROR: {str(e)}"

    resultados_chatbot.append({
        "numero_pedido": pedido["numero_pedido"],
        "estado_real": pedido["estado_espanol"],
        "respuesta_modelo": texto_respuesta
    })

# Verificar cuántos resultados alcanzaron a guardarse

print("Resultados almacenados:", len(resultados_chatbot))

if len(resultados_chatbot) > 0:
    display(pd.DataFrame(resultados_chatbot))
else:
    print("No se alcanzaron a guardar resultados.")

## 11. Resultados - Ejercicio 1

In [ ]:
# Guardar los resultados obtenidos hasta este momento

resultados_df = pd.DataFrame(resultados_chatbot)

archivo_resultados = RUTA_SALIDA / "Resultados_ModeloLocal_Ejercicio_1.xlsx"

with pd.ExcelWriter(archivo_resultados, engine="openpyxl") as writer:
    resultados_df.to_excel(
        writer,
        sheet_name="Resultados_ModeloLocal",
        index=False
    )

print("Resultados guardados correctamente.")
print("Pedidos procesados:", len(resultados_df))
print("Archivo:", archivo_resultados)

In [ ]:
# Evaluar si el modelo local respetó el estado real de los pedidos

def contiene_estado_correcto(fila):
    estado = fila["estado_real"].lower()
    respuesta = fila["respuesta_modelo"].lower()

    # Verificar que el estado real aparezca en la respuesta
    return estado in respuesta


resultados_df["estado_correcto"] = resultados_df.apply(
    contiene_estado_correcto,
    axis=1
)

print("Evaluación de los resultados:")
display(
    resultados_df[
        [
            "numero_pedido",
            "estado_real",
            "estado_correcto"
        ]
    ]
)

print("\nResumen:")
print(
    "Respuestas que respetaron el estado real:",
    resultados_df["estado_correcto"].sum(),
    "de",
    len(resultados_df)
)

In [ ]:
# Resumen de evaluación del Ejercicio 1

total_evaluados = len(resultados_df)
correctos = resultados_df["estado_correcto"].sum()
porcentaje_correctos = (correctos / total_evaluados) * 100

print("EVALUACIÓN DEL CHATBOT - EJERCICIO 1")
print("=" * 50)
print(f"Pedidos evaluados: {total_evaluados}")
print(f"Estados respetados: {correctos}")
print(f"Porcentaje: {porcentaje_correctos:.1f}%")
print()
print("Nota: esta métrica verifica únicamente si la respuesta")
print("contiene el estado real registrado en la base de datos.")

## 10. Pruebas de los prompts - Ejercicio 2 (devoluciones)

In [ ]:
# ============================================
# EJERCICIO 2 - PRUEBA CON EL MODELO LOCAL (OLLAMA)
# ============================================

resultados_devoluciones = []

for _, caso in df_devoluciones.iterrows():

    datos_producto = (
        f"Producto: {caso['producto']}\n"
        f"Categoría: {caso['categoria']}"
    )

    prompt_final = prompt_devolucion.format(
        datos_producto=datos_producto,
        motivo_devolucion=caso["motivo"]
    )

    try:
        respuesta = cliente.chat.completions.create(
            model=MODELO_LOCAL,
            messages=[{"role": "user", "content": prompt_final}]
        )

        texto_respuesta = respuesta.choices[0].message.content

    except Exception as e:
        texto_respuesta = f"ERROR: {str(e)}"

    resultados_devoluciones.append({
        "producto": caso["producto"],
        "categoria": caso["categoria"],
        "motivo": caso["motivo"],
        "puede_devolver_esperado": caso["puede_devolver"],
        "respuesta_modelo": texto_respuesta
    })

    print("Caso procesado:", caso["producto"])


resultados_devoluciones_df = pd.DataFrame(resultados_devoluciones)

print("\n============================================")
print("RESULTADOS DEL EJERCICIO 2")
print("============================================")

display(
    resultados_devoluciones_df[
        [
            "producto",
            "categoria",
            "puede_devolver_esperado",
            "respuesta_modelo"
        ]
    ]
)

## 11. Resultados - Ejercicio 2

In [ ]:
# Guardar los resultados del Ejercicio 2

archivo_resultados_devoluciones = RUTA_SALIDA / "Resultados_ModeloLocal_Ejercicio_2.xlsx"

with pd.ExcelWriter(archivo_resultados_devoluciones, engine="openpyxl") as writer:
    resultados_devoluciones_df.to_excel(
        writer,
        sheet_name="Resultados_ModeloLocal",
        index=False
    )

print("Resultados guardados correctamente.")
print("Casos procesados:", len(resultados_devoluciones_df))
print("Archivo:", archivo_resultados_devoluciones)

## 12. Conclusiones

- El chatbot construido con un **modelo open-source local (Ollama)** logra
  responder consultas de estado de pedido y de devoluciones a partir,
  exclusivamente, de la información proporcionada en el prompt (sin
  inventar datos).
- Usar un modelo local evita costos de API y problemas de créditos/billing,
  a cambio de una calidad de respuesta que depende del tamaño del modelo
  descargado (puedes probar con modelos más grandes si tu equipo lo permite).
- Los resultados cuantitativos de los dos ejercicios quedan guardados en la
  carpeta `Salida/` (`Resultados_ModeloLocal_Ejercicio_1.xlsx` y
  `Resultados_ModeloLocal_Ejercicio_2.xlsx`) para su revisión.
- La discusión de las Fases 1 y 2 del taller (selección/justificación del
  modelo y evaluación de fortalezas, limitaciones y riesgos éticos) se
  documenta por separado en formato Markdown, tal como lo pide la rúbrica
  de entrega.